# Adaptive Multiscale Spectro-Topological (AMST) Shape Descriptor
## MPEG-7 CE-Shape-1 Evaluation — v13 Final

**Architecture:** 5-component AMST + Stacking Ensemble vs. 8 baselines
- C1: Multi-scale Spectral (Fourier + wavelet + angle distribution)
- C2: Multi-resolution Shape Context
- C3: SPD Manifold Covariance (d=22 Log-Euclidean)
- C4: Deep Transfer Features (pretrained MobileNetV2, PCA 128d)
- C5: Geometric Complexity (convexity, curvature, contour stats)
- **Classifier**: Stacking ensemble SVM+RF+XGBoost with LogisticRegression meta-learner
- **Total**: 701 raw dims -> PCA 500 -> ensemble

In [ ]:
# Cell 1: Install
!pip install -q PyWavelets ripser persim xgboost scikit-image scikit-learn matplotlib seaborn scipy numpy pandas tqdm

import os, sys, warnings, json, copy, re, gc
warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['TORCH_HOME'] = '/content/.torch'
print('Done.')

In [ ]:
# Cell 2: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import scipy, scipy.stats, scipy.special
from scipy import ndimage
from scipy.interpolate import interp1d, splprep, splev
from scipy.spatial import ConvexHull
import pywt
print(f'PyWavelets: {pywt.__version__}')

from ripser import ripser as ripser_fn
RIPSER_OK = True
print('Ripser: OK')

try:
    import xgboost as xgb
    XGB_OK = True
    print(f'XGBoost: {xgb.__version__}')
except:
    XGB_OK = False
    print('XGBoost: not available')

from skimage import io, color, transform, feature, measure, img_as_float
from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects
from skimage.measure import find_contours, regionprops, label

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
print(f'PyTorch: {torch.__version__}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
N_SPLITS = 5
print(f'Seed={SEED} | Folds={N_SPLITS}')

In [ ]:
# Cell 3: Load MPEG-7 CE-Shape-1
DATA = Path('/content/mpeg7_shapes')
if not DATA.exists():
    import zipfile
    z = Path('/content/MPEG7_CE-Shape-1_Part_B.zip')
    if z.exists():
        with zipfile.ZipFile(str(z), 'r') as zf: zf.extractall('/content/mpeg7_shapes')
    else:
        raise FileNotFoundError('Upload MPEG7_CE-Shape-1_Part_B.zip to /content/')

img_dir = DATA / 'MPEG7_CE-Shape-1_Part_B'
if not img_dir.exists(): img_dir = DATA
EXT = {'.gif','.png','.jpg'}
all_files = sorted([f for f in img_dir.rglob('*') if f.suffix.lower() in EXT])
all_files = [f for f in all_files if f.stem.lower() not in ('confusions','shapedata')]
print(f'Files: {len(all_files)}')

def parse_label(fp):
    s = fp.stem
    return re.sub(r'[-_]?\d+$', '', s).strip('-_').lower()

samples = [(f, parse_label(f)) for f in all_files]
classes = sorted(set(s[1] for s in samples))
print(f'Samples: {len(samples)} | Classes: {len(classes)}')

In [ ]:
# Cell 4: Preprocessing
IMG_SIZE = (64, 64)

def load_bin(path):
    img = io.imread(str(path))
    img = np.squeeze(img)
    if img.ndim == 3:
        gray = color.rgb2gray(img[...,:3] if img.shape[2]==4 else img)
    elif img.ndim == 2:
        gray = img_as_float(img)
    else:
        raise ValueError(f'Unexpected shape: {img.shape}')
    gray = transform.resize(gray, IMG_SIZE, anti_aliasing=True)
    try: t = threshold_otsu(gray)
    except: t = 0.5
    b = gray < t
    if b.sum() < IMG_SIZE[0]*IMG_SIZE[1]*0.02: b = ~b
    b = binary_closing(b, disk(2))
    b = binary_opening(b, disk(1))
    b = remove_small_objects(b.astype(bool), min_size=50)
    return b.astype(np.uint8)

def get_contour(b, n=128):
    cl = find_contours(b.astype(float), 0.5)
    if not cl: return np.zeros((n,2))
    c = max(cl, key=len)
    try:
        tck, u = splprep([c[:,0], c[:,1]], s=0, per=True)
        u_new = np.linspace(0, 1, n, endpoint=False)
        x_s, y_s = splev(u_new, tck)
        return np.column_stack([x_s, y_s])
    except:
        d = np.diff(c, axis=0)
        arc = np.r_[0, np.cumsum(np.sqrt((d**2).sum(axis=1)))]
        if arc[-1] < 1e-8: return np.zeros((n,2))
        u = np.linspace(0, arc[-1], n, endpoint=False)
        return np.column_stack([np.interp(u, arc, c[:,0]), np.interp(u, arc, c[:,1])])

def center_scale(c):
    c = c - c.mean(axis=0)
    r = np.sqrt((c**2).sum(axis=1)).max()
    return c / r if r > 1e-8 else c

def curvature(c):
    x, y = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(y); x2=np.gradient(x1); y2=np.gradient(y1)
    return (x1*y2 - x2*y1) / (x1**2 + y1**2 + 1e-12)**1.5

print('Loading shapes...')
bins, cnts, labs = [], [], []
for path, label in tqdm(samples):
    try:
        b = load_bin(path)
        c = center_scale(get_contour(b))
        bins.append(b); cnts.append(c); labs.append(label)
    except Exception as e:
        print(f'Fail: {path.name} - {e}')

le = LabelEncoder()
y = le.fit_transform([str(l) for l in labs])
n_cls = len(le.classes_)
N = len(cnts)
print(f'Loaded: {N} | Classes: {n_cls}')

In [ ]:
# Cell 5: Baseline Descriptors
def fourier_desc(c, K=64):
    r = np.sqrt((c**2).sum(axis=1))
    F = np.fft.fft(r); mag = np.abs(F)
    d = mag[1] if mag[1]>1e-8 else mag.max()+1e-12
    mag_n = mag / d
    return np.concatenate([mag_n[1:K+1][:-1], np.angle(F)[1:17]])

def wavelet_desc(c):
    r = np.sqrt((c**2).sum(axis=1)); r -= r.mean()
    f = []
    for w in ['db4','haar','sym4']:
        lv = max(1, min(5, pywt.dwt_max_level(len(r), w)))
        coeffs = pywt.wavedec(r, w, level=lv, mode='periodization')
        en = np.array([np.sum(co**2) for co in coeffs]); en /= en.sum()+1e-12
        f.append(en)
    mx = max(len(e) for e in f)
    return np.concatenate([np.pad(e, (0,mx-len(e))) for e in f])

def hybrid_desc(c): return np.concatenate([fourier_desc(c,64), wavelet_desc(c)])

def zernike_moments(b, order=10):
    h,w = b.shape
    yg,xg = np.mgrid[-1:1:1j*h, -1:1:1j*w]
    r = np.sqrt(xg**2+yg**2); theta = np.arctan2(yg,xg)
    mask = (r<=1.) & (b>0); moms = []
    for n in range(order+1):
        for m in range(-n, n+1, 2):
            if (n-abs(m))%2 != 0: continue
            R = np.zeros_like(r)
            for s in range((n-abs(m))//2+1):
                c = ((-1)**s * scipy.special.factorial(n-s)) / (
                    scipy.special.factorial(s) *
                    scipy.special.factorial((n+abs(m))//2-s) *
                    scipy.special.factorial((n-abs(m))//2-s) + 1e-300)
                R += c * r**(n-2*s)
            V = R * np.exp(-1j*m*theta)
            moms.append(np.abs(np.sum(V[mask]*b[mask])*(n+1)/np.pi))
    return np.array(moms[:36])

def shape_context(c, nr=5, nt=12):
    N = len(c); step = max(1, N//64); pts = c[::step]; n = len(pts)
    dx = pts[:,1:2]-pts[np.newaxis,:,1]; dy = pts[:,0:1]-pts[np.newaxis,:,0]
    dist = np.sqrt(dx**2+dy**2+1e-12); ang = np.arctan2(dy, dx)
    ld = np.log(dist/(dist.max()+1e-12)+1e-12)
    rb = np.linspace(ld.min()-0.01, 0.01, nr+1)
    tb = np.linspace(-np.pi, np.pi, nt+1)
    Hg = np.zeros(nr*nt)
    for i in range(n):
        mi = np.arange(n)!=i
        H,_,_ = np.histogram2d(ld[i,mi], ang[i,mi], bins=[rb,tb])
        Hg += H.flatten()
    return Hg/(Hg.sum()+1e-12)

def css_desc(c, sigmas=None):
    if sigmas is None: sigmas = [1,2,4,8,16,32,64,128,256,512]
    x,yc = c[:,1], c[:,0]; f = []
    for s in sigmas:
        xs = ndimage.gaussian_filter1d(x, s, mode='wrap')
        ys = ndimage.gaussian_filter1d(yc, s, mode='wrap')
        x1=np.gradient(xs); x2=np.gradient(x1); y1=np.gradient(ys); y2=np.gradient(y1)
        k = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
        f += [float(np.sum(np.diff(np.sign(k))!=0)), float(np.mean(np.abs(k)))]
    return np.array(f)

def hog_desc(b):
    return feature.hog(b.astype(np.float32), orientations=9,
                       pixels_per_cell=(8,8), cells_per_block=(1,1), feature_vector=True)

print('Baselines defined.')

In [ ]:
# Cell 6: Deep Feature Extractor (MobileNetV2)
print('Loading pretrained MobileNetV2...')
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model = nn.Sequential(*list(model.children())[:-1])
model.eval().to(DEVICE)
for p in model.parameters(): p.requires_grad = False
print(f'MobileNetV2 loaded on {DEVICE}')

transform_deep = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_deep(b):
    x = (b * 255).astype(np.uint8)
    x = np.stack([x]*3, axis=-1)
    x = transform_deep(x).unsqueeze(0).to(DEVICE)
    feat = model(x).cpu().numpy().flatten()
    return feat

print('Extracting deep features...')
deep_feats = []
for i in tqdm(range(N)):
    deep_feats.append(extract_deep(bins[i]))
X_deep_raw = np.array(deep_feats)
print(f'Deep features: {X_deep_raw.shape}')

# PCA reduce deep features
pca_deep = PCA(n_components=min(128, N))
X_deep = pca_deep.fit_transform(X_deep_raw)
print(f'Deep features (PCA 128): {X_deep.shape}, variance ratio: {pca_deep.explained_variance_ratio_.sum():.3f}')

In [ ]:
# Cell 7: Baseline Feature Extraction
print('Extracting baseline features...')
FD, WD, HY, ZE, CS, SC, HG = [],[],[],[],[],[],[]
for i in tqdm(range(N)):
    c=cnts[i]; b=bins[i]
    try: FD.append(fourier_desc(c))
    except: FD.append(np.zeros(79))
    try: WD.append(wavelet_desc(c))
    except: WD.append(np.zeros(18))
    try: HY.append(hybrid_desc(c))
    except: HY.append(np.zeros(97))
    try: ZE.append(zernike_moments(b))
    except: ZE.append(np.zeros(36))
    try: CS.append(css_desc(c))
    except: CS.append(np.zeros(20))
    try: SC.append(shape_context(c))
    except: SC.append(np.zeros(60))
    try: HG.append(hog_desc(b))
    except: HG.append(np.zeros(576))

X_fd=np.array(FD); X_wd=np.array(WD); X_hy=np.array(HY)
X_ze=np.array(ZE); X_cs=np.array(CS); X_sc=np.array(SC)
X_hg=np.array(HG)

for nm,X in [('FD',X_fd),('WD',X_wd),('HY',X_hy),('ZE',X_ze),
             ('CS',X_cs),('SC',X_sc),('HG',X_hg),('Deep',X_deep)]:
    print(f'  {nm:4s}: {X.shape}')
gc.collect()

In [ ]:
# Cell 8: AMST v13 Components
def c1_spectral(c):
    r = np.sqrt((c**2).sum(axis=1))
    F = np.fft.fft(r); mag = np.abs(F); ph = np.angle(F)
    d = mag[1] if mag[1]>1e-8 else mag.max()+1e-12
    fd = mag/d; fd = fd[1:81]
    phs = ph[1:17]
    x,yc = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    kc = kappa - kappa.mean()
    max_lv = pywt.dwt_max_level(len(kc), 'db4')
    L = max(2, min(5, max_lv))
    coeffs = pywt.wavedec(kc, 'db4', level=L, mode='periodization') if L>1 else pywt.wavedec(kc, 'db1', level=2, mode='periodization')
    energies = np.array([np.sum(cf**2) for cf in coeffs])
    E = energies / (energies.sum()+1e-12)
    xi = np.linspace(0, 1, len(E)); xo = np.linspace(0, 1, 32)
    E32 = interp1d(xi, E, kind='linear', fill_value='extrapolate')(xo)
    E32 = np.maximum(E32, 0); E32 /= E32.sum()+1e-12
    angles = np.arctan2(c[:,0], c[:,1])
    ah,_ = np.histogram(angles, bins=32, range=(-np.pi, np.pi))
    return np.concatenate([fd, phs, E32, ah/ah.sum()])

def c2_shape_context(c):
    feat = []
    for n_pts in [64, 128]:
        step = max(1, len(c)//n_pts)
        pts = c[::step][:n_pts]; n = len(pts)
        dx = pts[:,1:2]-pts[np.newaxis,:,1]; dy = pts[:,0:1]-pts[np.newaxis,:,0]
        dist = np.sqrt(dx**2+dy**2+1e-12); ang = np.arctan2(dy, dx)
        ld = np.log(dist/(dist.max()+1e-12)+1e-12)
        rb = np.linspace(ld.min()-0.01, 0.01, 5+1)
        tb = np.linspace(-np.pi, np.pi, 12+1)
        Hg = np.zeros(60)
        for i in range(n):
            mi = np.arange(n)!=i
            H,_,_ = np.histogram2d(ld[i,mi], ang[i,mi], bins=[rb,tb])
            Hg += H.flatten()
        feat.append(Hg/(Hg.sum()+1e-12))
    return np.concatenate(feat)

def c3_spd(c):
    r = np.sqrt((c**2).sum(axis=1)); N=len(r)
    x,yc = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    t = np.linspace(0, 2*np.pi, N, endpoint=False)
    rows = [r-r.mean(), x-x.mean(), yc-yc.mean(), kappa, np.cos(t), np.sin(t),
            ndimage.gaussian_filter1d(r-r.mean(),2,mode='wrap'),
            ndimage.gaussian_filter1d(r-r.mean(),8,mode='wrap'),
            ndimage.gaussian_filter1d(kappa,2,mode='wrap'),
            ndimage.gaussian_filter1d(kappa,4,mode='wrap'),
            ndimage.gaussian_filter1d(kappa,8,mode='wrap'),
            np.gradient(kappa),
            ndimage.gaussian_filter1d(r-r.mean(),4,mode='wrap'),
            ndimage.gaussian_filter1d(kappa,1,mode='wrap'),
            np.abs(kappa), kappa**2, np.sqrt(np.abs(kappa)+1e-12),
            np.sin(2*t), np.cos(2*t), np.arctan2(yc,x),
            ndimage.gaussian_filter1d(np.abs(kappa),2,mode='wrap'),
            np.sin(3*t)]
    fm = np.array(rows[:22], dtype=float)
    fm -= fm.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(fm, axis=1, keepdims=True)
    fm /= (norms+1e-12)
    S = (fm @ fm.T)/(N-1) + 1e-4*np.eye(22)
    ev,evec = np.linalg.eigh(S)
    ev = np.maximum(ev, 1e-8)
    logS = evec @ np.diag(np.log(ev)) @ evec.T
    return logS[np.triu_indices(22)]

def c5_complexity(c, b):
    f = []
    hull = ConvexHull(c)
    ha = hull.volume; hp = hull.area
    ca = np.abs(np.sum(c[:-1,0]*c[1:,1]-c[1:,0]*c[:-1,1]))/2
    cp = np.sum(np.sqrt(np.diff(c[:,0])**2+np.diff(c[:,1])**2))
    f += [hp/(cp+1e-12), ca/(ha+1e-12), 4*np.pi*ca/(cp**2+1e-12)]
    labeled = measure.label(b); props = regionprops(labeled)
    if props:
        p=props[0]; f += [float(p.euler_number), p.major_axis_length/(p.minor_axis_length+1e-12),
                          p.extent, p.eccentricity, p.equivalent_diameter_area/max(b.shape),
                          p.perimeter/(cp+1e-12), p.area/(b.shape[0]*b.shape[1])]
    else: f += [0.0]*7
    x,yc=c[:,1],c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    f += [np.mean(kappa), np.std(kappa), np.max(np.abs(kappa)),
          np.sum(np.diff(np.sign(kappa))!=0)/len(kappa),
          float(scipy.stats.skew(kappa)), float(scipy.stats.kurtosis(kappa)),
          float(np.percentile(np.abs(kappa), 90))]
    d = np.sqrt((c**2).sum(axis=1))
    f += [np.std(d), float(np.max(d)-np.min(d)), np.mean(d),
          float(np.percentile(d,25)), float(np.percentile(d,75)),
          float(np.percentile(d,10)), float(np.percentile(d,90)),
          np.sum(kappa > 0)/len(kappa), np.sum(kappa < 0)/len(kappa)]
    f += [np.max(d)/np.min(d+1e-12), np.sqrt(np.mean(d**2))]
    return np.array(f, dtype=float)

print('Components defined.')

In [ ]:
# Cell 9: AMST Feature Extraction
print('Extracting AMST features...')
AM_C1, AM_C2, AM_C3, AM_C5 = [],[],[],[]
for i in tqdm(range(N)):
    c=cnts[i]; b=bins[i]
    try:
        AM_C1.append(c1_spectral(c))
        AM_C2.append(c2_shape_context(c))
        AM_C3.append(c3_spd(c))
        AM_C5.append(c5_complexity(c,b))
    except:
        AM_C1.append(np.zeros(160))
        AM_C2.append(np.zeros(120))
        AM_C3.append(np.zeros(253))
        AM_C5.append(np.zeros(40))

AM_c1=np.array(AM_C1); AM_c2=np.array(AM_C2); AM_c3=np.array(AM_C3); AM_c5=np.array(AM_C5)
print(f'C1:{AM_c1.shape} C2:{AM_c2.shape} C3:{AM_c3.shape} C4:{X_deep.shape} C5:{AM_c5.shape}')

# Standardize & concatenate
am_parts = []
for Xp in [AM_c1, AM_c2, AM_c3, X_deep, AM_c5]:
    Xp_s = (Xp - Xp.mean(0)) / (Xp.std(0).clip(1e-12, None))
    am_parts.append(np.nan_to_num(Xp_s))

X_am_raw = np.concatenate(am_parts, axis=1)
print(f'AMST raw dim: {X_am_raw.shape[1]}')

# PCA to 500
pca_am = PCA(n_components=500)
X_am = pca_am.fit_transform(X_am_raw)
print(f'AMST PCA dim: {X_am.shape[1]}, variance ratio: {pca_am.explained_variance_ratio_.sum():.3f}')

In [ ]:
# Cell 10: Classification
def best_svm(X_tr, y_tr):
    param_grid = {
        'C': [0.01, 0.1, 1, 10, 100, 1000, 10000],
        'gamma': ['scale', 'auto', 1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]
    }
    gs = GridSearchCV(SVC(kernel='rbf', decision_function_shape='ovr', class_weight='balanced'),
                      param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr); return gs.best_params_

def eval_method(X, y, name):
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; faccs=[]
    for tr, te in skf.split(X, y):
        X_tr,X_te=X[tr],X[te]; y_tr,y_te=y[tr],y[te]
        sc = RobustScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)
        bp = best_svm(X_tr_s, y_tr)
        clf = SVC(kernel='rbf', decision_function_shape='ovr', class_weight='balanced', **bp)
        clf.fit(X_tr_s,y_tr); yp=clf.predict(X_te_s)
        accs.append(accuracy_score(y_te,yp))
        f1s.append(f1_score(y_te,yp,average='macro',zero_division=0))
        faccs.append(accuracy_score(y_te,yp))
    return {'Method':name,'Accuracy':np.mean(accs),'Acc_std':np.std(accs),'F1':np.mean(f1s),'Dim':X.shape[1],'faccs':faccs}

def eval_amst_stack(X, y, name):
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; faccs=[]
    for tr, te in skf.split(X, y):
        X_tr,X_te=X[tr],X[te]; y_tr,y_te=y[tr],y[te]
        sc = RobustScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)
        rf = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=SEED, n_jobs=-1)
        xgb_c = xgb.XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.1,
                                   random_state=SEED, n_jobs=-1, verbosity=0,
                                   eval_metric='mlogloss') if XGB_OK else None
        rf_pred = cross_val_predict(rf, X_tr_s, y_tr, cv=3, method='predict_proba')
        if xgb_c is not None:
            xgb_pred = cross_val_predict(xgb_c, X_tr_s, y_tr, cv=3, method='predict_proba')
            meta_tr = np.column_stack([rf_pred, xgb_pred])
        else:
            meta_tr = rf_pred
        rf.fit(X_tr_s, y_tr); rf_te = rf.predict_proba(X_te_s)
        if xgb_c is not None:
            xgb_c.fit(X_tr_s, y_tr); meta_te = np.column_stack([rf_te, xgb_c.predict_proba(X_te_s)])
        else:
            meta_te = rf_te
        meta = LogisticRegression(multi_class='multinomial', max_iter=5000, C=1.0, random_state=SEED)
        meta.fit(meta_tr, y_tr)
        yp = meta.predict(meta_te)
        accs.append(accuracy_score(y_te,yp))
        f1s.append(f1_score(y_te,yp,average='macro',zero_division=0))
        faccs.append(accuracy_score(y_te,yp))
    return {'Method':name,'Accuracy':np.mean(accs),'Acc_std':np.std(accs),'F1':np.mean(f1s),'faccs':faccs}

print('Running classification...')
res = []
for Xf, nm in [(X_fd,'Fourier'),(X_wd,'Wavelet'),(X_hy,'Hybrid'),
               (X_ze,'Zernike'),(X_cs,'CSS'),(X_sc,'ShapeCtx'),
               (X_hg,'HOG'),(X_deep,'CNN_Feat')]:
    r = eval_method(Xf, y, nm)
    res.append(r)
    print(f'{nm:12s} | Acc:{r["Accuracy"]*100:.2f}% F1:{r["F1"]*100:.2f}%')

print('\nAMST Stacking Ensemble...')
r_amst = eval_amst_stack(X_am, y, 'AMST')
res.append(r_amst)
print(f'{"AMST":12s} | Acc:{r_amst["Accuracy"]*100:.2f}% F1:{r_amst["F1"]*100:.2f}%')

df = pd.DataFrame(res)
amst_acc = df[df.Method=='AMST']['Accuracy'].values[0]*100
best_base = df[df.Method!='AMST']['Accuracy'].max()*100
print(f'\nAMST: {amst_acc:.2f}% | Best base: {best_base:.2f}% | Gain: +{amst_acc-best_base:.2f} pp')

print('\nAll results:')
print(df[['Method','Accuracy','F1','Dim']].to_string(index=False))

In [ ]:
# Cell 11: Statistical Significance
amst_f = np.array(df[df.Method=='AMST']['faccs'].values[0])
N_CMP = len(df)-1
ALPHA_BONF = 0.05/N_CMP
print(f'Bonferroni: {N_CMP} comparisons, alpha={ALPHA_BONF:.6f}')
print(f'{"Method":<14} {"AMST%":>8} {"Base%":>8} {"Delta":>8} {"p-val":>10} {"Sig?":>8}')
print('-'*50)
rows = []
for _, r in df.iterrows():
    nm = r.Method
    if nm=='AMST': continue
    bf = np.array(r['faccs'])
    t, p = scipy.stats.ttest_rel(amst_f, bf)
    d = (amst_f.mean()-bf.mean())*100
    sb = '*' if p<ALPHA_BONF else ' '
    print(f'{nm:14s} {amst_f.mean()*100:>8.2f} {bf.mean()*100:>8.2f} {d:>+8.2f} {p:>10.5f} {sb:>8}')
    rows.append({'Baseline':nm, 'Delta_pp':d, 'p_value':p, 'Bonf_sig':p<ALPHA_BONF})
stat_df = pd.DataFrame(rows)
print(f'\nSig wins (Bonferroni): {stat_df.Bonf_sig.sum()}/{N_CMP}')

In [ ]:
# Cell 12: Retrieval
def bullseye(X, y, K=40):
    Xs=StandardScaler().fit_transform(np.nan_to_num(X))
    N=len(y); good=0; poss=0
    for qi in range(N):
        d=np.sqrt(((Xs-Xs[qi])**2).sum(1))
        rk=np.argsort(d); rk=rk[rk!=qi]
        matches=(y[rk[:K]]==y[qi]).sum()
        n_same=(y==y[qi]).sum()-1
        good+=matches; poss+=min(K,n_same)
    return good/poss*100
def map_score(X, y):
    Xs=StandardScaler().fit_transform(np.nan_to_num(X))
    N=len(y); APs=[]
    for qi in range(N):
        d=np.sqrt(((Xs-Xs[qi])**2).sum(1))
        rk=np.argsort(d); rk=rk[rk!=qi]
        rel=(y[rk]==y[qi]).astype(int)
        if rel.sum()==0: continue
        cs=np.cumsum(rel); pos=np.arange(1,len(rk)+1)
        APs.append((cs/pos*rel).sum()/rel.sum())
    return np.mean(APs)

print('Retrieval...')
ret_m = [(X_fd,'Fourier'),(X_wd,'Wavelet'),(X_hy,'Hybrid'),
         (X_ze,'Zernike'),(X_sc,'ShapeCtx'),(X_deep,'CNN_Feat'),(X_am,'AMST')]
ret_r=[]
for Xf,nm in ret_m:
    be=bullseye(Xf,y,40); mp=map_score(Xf,y)
    ret_r.append({'Method':nm,'Bullseye':be,'MAP':mp})
    print(f'{nm:12s} Bullseye:{be:.1f}% MAP:{mp:.4f}')
rdf=pd.DataFrame(ret_r)
print(f'\nAMST Bullseye: {rdf[rdf.Method=="AMST"]["Bullseye"].values[0]:.1f}%')
print(f'Best baseline: {rdf[rdf.Method!="AMST"]["Bullseye"].max():.1f}%')

In [ ]:
# Cell 13: Figures
methods = df.Method.values
accs = df.Accuracy.values*100; stds = df.Acc_std.values*100; f1s = df.F1.values*100
colors = ['#5B7FA6']*8+['#E84040']

fig,axes = plt.subplots(1,2,figsize=(16,6))
for ax,vals,title in [(axes[0],accs,'Accuracy (%)'),(axes[1],f1s,'F1-Score (%)')]:
    bars=ax.barh(methods,vals,xerr=stds if ax==axes[0] else None,
                 color=colors,edgecolor='white',capsize=4,height=0.65)
    ax.set_xlim(0,110); ax.set_xlabel('%'); ax.set_title(title); ax.grid(axis='x',alpha=0.3)
    for i,(b,v) in enumerate(zip(bars,vals)):
        fw='bold' if i==len(methods)-1 else 'normal'
        ax.text(v+0.5,b.get_y()+b.get_height()/2,f'{v:.1f}%',va='center',fontsize=8,fontweight=fw)
plt.tight_layout()
plt.savefig('/content/fig_classification.png',dpi=120,bbox_inches='tight')
plt.show(); print('Fig saved.')

fig,ax=plt.subplots(figsize=(10,4))
rdf_s=rdf.sort_values('Bullseye')
ax.barh(rdf_s['Method'],rdf_s['Bullseye'],
        color=['#E84040' if 'AMST' in n else '#5B7FA6' for n in rdf_s['Method']])
ax.set_title('Bullseye Rating (%)'); ax.set_xlim(0,105); ax.grid(axis='x',alpha=0.3)
plt.tight_layout(); plt.savefig('/content/fig_bullseye.png',dpi=120,bbox_inches='tight')
plt.show()

In [ ]:
# Cell 14: Summary
print('='*60)
print('AMST v13 FINAL RESULTS')
print('='*60)
print(f'Dataset: MPEG-7 CE-Shape-1 ({N} images, {n_cls} classes)')
print(f'CV: {N_SPLITS}-fold stratified')
print(f'AMST dim: {X_am.shape[1]} (PCA from {X_am_raw.shape[1]})')
print(f'AMST Acc: {amst_acc:.2f}%')
print(f'Best baseline: {best_base:.2f}%')
print(f'Gain: +{amst_acc-best_base:.2f} pp')
print(f'Bonferroni sig wins: {stat_df.Bonf_sig.sum()}/{N_CMP}')
be_amst = rdf[rdf.Method=='AMST']['Bullseye'].values[0]
be_best = rdf[rdf.Method!='AMST']['Bullseye'].max()
print(f'Bullseye: AMST={be_amst:.1f}% vs best baseline={be_best:.1f}%')
print('='*60)